# Recommendation experiments

This notebook produces the recommendation results of the paper (Section 6.3), in the order of the protocol:

1. **Grid search** of the hyperparameters of each model on each dataset (seed 0), with early stopping on NDCG@5.
2. **Five seeds**: every model is trained and evaluated again with its best hyperparameters, with seeds 0 to 4. KNN is deterministic and is evaluated once.
3. **Results**: mean and standard deviation of each metric across seeds, and paired tests between models.

| In the paper | In this notebook |
|---|---|
| Table *Hyperparameters tested* | `GRID`, section *Hyperparameters* |
| Appendix table *Best hyperparameters* | `BEST` and `KNN_BEST_N`, section *Hyperparameters* |
| Table *Evaluation results* (mean and standard deviation) | section *Results* |
| Table *Relative difference in NDCG@5* (paired tests) | section *Results* |

The figure on the number of layers comes from `d_Graphic.ipynb`, and the table on the length of the encoded texts from `e_TokenLengths.ipynb`.

**Execution.** The full grid search took more than four weeks, so it is off by default (`RUN_GRID_SEARCH`): `BEST` already holds its outcome. A run whose `per_user_metrics.pkl` exists is skipped, so the notebook can be executed again after an interruption, and the section *Results* only reads the saved runs.

Seed 0 gives the same figures as the grid search for ALS, LightGCN and KNN. BPR and LMF are not reproducible exactly, because the multithreaded SGD of `implicit` is not deterministic.

**Outputs.** For each run `<dataset>_<model>_seed<k>`, the folder `results/recommendation_runs/<run>/` contains `evaluation_results.csv` (metrics and best hyperparameters of each model) and `per_user_metrics.pkl` (metrics of every test user, used by the paired tests). The runs of the grid search are in the subfolder `grid_search/`, and the log of each run in `scratch/recommendation_logs/`.

**Requirements.** The environment of `recommender_system/pyproject.toml` (see the README), and a GPU for LightGCN. On a 24 GB GPU, the LightGCN runs take from a few minutes (ml-100k) to several hours (Goodreads, Tomplay) each. Runs are executed in child processes started with `fork`, so this notebook requires Linux or macOS.

## Imports and paths

TensorFlow and `implicit` are imported in the child process of each run, once its GPU is selected (`import_models`, section *One run*).

In [ ]:
import ast
import copy
import itertools
import multiprocessing
import os
import pickle
import random
import re
import sys
import threading
import time
import traceback
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sparse
from huggingface_hub import snapshot_download
from scipy.stats import wilcoxon
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm  # Text bars: this code also runs in child processes, whose output goes to the log
from tqdm.auto import tqdm as tqdm_auto

# The notebooks are executed from recommender_system/
REPO = Path.cwd().resolve().parent
assert (REPO / "recommender_system" / "c_Recommendations.ipynb").exists(), "Run this notebook from recommender_system/"
DATA = REPO / "results" / "recommendation"
OUT_ROOT = REPO / "results" / "recommendation_runs"
# Preprocessed datasets and LightGCN checkpoints (large, not versioned)
CACHE_ROOT = REPO / "scratch" / "recommendation_cache"
CHECKPOINT_ROOT = REPO / "scratch" / "recommendation_checkpoints"
LOG_DIR = REPO / "scratch" / "recommendation_logs"

## Settings

* `SEEDS`: random seeds of the stochastic models.
* `DATASETS`: datasets to use (`lingorank` is the Zeeguu dataset). Tomplay is proprietary and not distributed: it is skipped if its files are missing.
* `GPUS`: GPUs used for LightGCN, one run at a time per GPU.
* `CPU_JOBS`: number of ALS/BPR/LMF/KNN runs executed in parallel (on CPU).
* `RUN_GRID_SEARCH`: run the grid search again before the final runs.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
DATASETS = ["ml_100k", "lingorank", "goodreads", "tomplay"]
GPUS = ["0"]
CPU_JOBS = 4
RUN_GRID_SEARCH = False

## Hyperparameters

`GRID` is the grid of the table of tested hyperparameters of the paper. `BEST` is its outcome, the best hyperparameters of each model on each dataset (Appendix of the paper), given as one-element grids. For LightGCN, `n` is the number of most recent items of the user profile averaged into the initial user embedding.

KNN is always evaluated on all the profile sizes `KNN_PROFILE_SIZES`, and `KNN_BEST_N` selects the size reported in the paper. It is the same profile size `n` as for LightGCN. In the saved results it appears as `k=<n>`.

In [ ]:
GRID = {
    "als": {
        "factors": [20,50,100],
        "regularization": [0.001, 0.01, 0.05, 0.1],
        "alpha": [1,5,10,40,50]
    },
    "bpr": {
        "factors": [20,50,100],
        "regularization": [0.001, 0.01, 0.05, 0.1],
        "learning_rate": [0.001, 0.01, 0.05, 0.1],
        "verify_negative_samples": [True, False]
    },
    "lmf": {
        "factors": [20,50,100],
        "regularization": [0.05, 0.1, 0.5, 0.6, 1],
        "learning_rate": [0.05, 0.1, 0.5, 1],
        "neg_prop": [10, 30, 50]
    },
    "lightgcn": {
        "n_layers": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "n": [1, 2, 3, 5, 20, 30, 50],
        "lr": [0.0005, 0.001, 0.01, 0.05, 0.1],
    },
}

KNN_PROFILE_SIZES = [1, 2, 3, 4, 5, 10, 20, 30, 50, 100]

BEST = {
    "lingorank": {
        "als": {"factors": [50], "regularization": [0.05], "alpha": [5]},
        "bpr": {"factors": [50], "regularization": [0.01], "learning_rate": [0.001], "verify_negative_samples": [False]},
        "lmf": {"factors": [20], "regularization": [1], "learning_rate": [0.1], "neg_prop": [10]},
        "ada": {"n_layers": [9], "n": [1], "lr": [0.0005]},
        "xavier_ada": {"n_layers": [10], "n": [1], "lr": [0.0005]},
        "bert": {"n_layers": [10], "n": [2], "lr": [0.01]},
        "xavier_bert": {"n_layers": [10], "n": [1], "lr": [0.001]},
    },
    "ml_100k": {
        "als": {"factors": [20], "regularization": [0.05], "alpha": [1]},
        "bpr": {"factors": [50], "regularization": [0.001], "learning_rate": [0.05], "verify_negative_samples": [True]},
        "lmf": {"factors": [100], "regularization": [1], "learning_rate": [0.05], "neg_prop": [50]},
        "ada": {"n_layers": [2], "n": [5], "lr": [0.001]},
        "xavier_ada": {"n_layers": [2], "n": [1], "lr": [0.001]},
        "bert": {"n_layers": [2], "n": [20], "lr": [0.001]},
        "xavier_bert": {"n_layers": [1], "n": [1], "lr": [0.001]},
    },
    "goodreads": {
        "als": {"factors": [100], "regularization": [0.01], "alpha": [40]},
        "bpr": {"factors": [100], "regularization": [0.01], "learning_rate": [0.001], "verify_negative_samples": [True]},
        "lmf": {"factors": [100], "regularization": [1], "learning_rate": [0.1], "neg_prop": [10]},
        "ada": {"n_layers": [2], "n": [2], "lr": [0.0005]},
        "xavier_ada": {"n_layers": [3], "n": [1], "lr": [0.001]},
        "bert": {"n_layers": [5], "n": [5], "lr": [0.001]},
        "xavier_bert": {"n_layers": [2], "n": [1], "lr": [0.001]},
    },
    "tomplay": {
        "als": {"factors": [100], "regularization": [0.001], "alpha": [40]},
        "bpr": {"factors": [50], "regularization": [0.001], "learning_rate": [0.05], "verify_negative_samples": [True]},
        "lmf": {"factors": [100], "regularization": [1], "learning_rate": [0.05], "neg_prop": [10]},
        "ada": {"n_layers": [2], "n": [1], "lr": [0.0005]},
        "xavier_ada": {"n_layers": [3], "n": [1], "lr": [0.0005]},
        "bert": {"n_layers": [2], "n": [1], "lr": [0.0005]},
        "xavier_bert": {"n_layers": [2], "n": [1], "lr": [0.0005]},
    },
}

# Dataset flag -> short name used in the run names
SHORT_NAMES = {"ml_100k": "ml-100k", "lingorank": "zeeguu", "goodreads": "goodreads", "tomplay": "tomplay"}
LIGHTGCN_VARIANTS = ["ada", "xavier_ada", "bert", "xavier_bert"]

# Best profile size of KNN (Appendix of the paper), per dataset key of `load_datasets`
KNN_BEST_N = {"ml-100k": {"ada": 2, "bert": 1}, "LingoRank": {"ada": 4, "bert": 4},
              "Goodreads": {"ada": 1, "bert": 1}, "Tomplay": {"ada": 1, "bert": 1}}

## Data

The public datasets (Zeeguu, ml-100k, Goodreads) are downloaded from the Hugging Face dataset `OloriBern/FLDE`, if they are not already in `results/recommendation/`. Tomplay is proprietary and not distributed.

The item embeddings are not part of this dataset: `b_GenerateEmbeddings.ipynb` writes them to `results/recommendation/embeddings_*.csv.gz` (ADA requires an OpenAI API key). LightGCN with ADA or BERT needs the corresponding embeddings and KNN uses the ones that exist; ALS, BPR, LMF and LightGCN with the Xavier initialisation need none. A run that cannot be executed is reported as `FAILED`, and the section *Results* uses the runs that exist.

In [ ]:
if not (DATA / "ml-100k" / "u.data").exists():
    snapshot_download(
        repo_id="OloriBern/FLDE",
        allow_patterns=["recommendation/*"],
        local_dir=os.path.join("..", "results"),
        revision="main",
        repo_type="dataset"
    )

if "tomplay" in DATASETS and not (DATA / "Tomplay" / "interactions.csv").exists():
    print("Tomplay data not found (proprietary dataset): Tomplay is skipped.")
    DATASETS = [d for d in DATASETS if d != "tomplay"]
print("Datasets:", DATASETS)

Configuration of a run: the flags and grids read by the code below (`config` holds the ones that are set).

In [ ]:
DATASET_FLAGS = ["lingorank", "ml_100k", "goodreads", "tomplay"]
MODEL_FLAGS = ["content_based", "als", "bpr", "lmf", "ada", "bert", "xavier_ada", "xavier_bert"]


def set_config(config, seed):
    """Set the global flags and the seed of a run."""
    global cf, lightgcn, my_seed
    for flag in DATASET_FLAGS + MODEL_FLAGS:
        globals()[flag] = bool(config.get(flag, False))
    for name in ["als_params", "bpr_params", "lmf_params", "lightgcn_params"]:
        globals()[name] = config.get(name)
    cf = als or bpr or lmf  # If at least one, we do the CF
    lightgcn = ada or bert or xavier_ada or xavier_bert  # If at least one, we do the LightGCN
    my_seed = seed
    random.seed(my_seed)
    np.random.seed(my_seed)

### Train-test split and loading of the datasets

In [ ]:
def split_user_data(group, test_size):
        if len(group) == 1:
            return pd.DataFrame(), pd.DataFrame(), group
        else:
            n = max(1, int(round(len(group) * test_size)))
            return group.iloc[:-n], group.iloc[-n:], pd.DataFrame()

In [ ]:
def train_test_split(df, n=None, test_size=0.2, one_core=False):
    # Sort by user_id and timestamp
    # df.sort_values(by=['user_id', 'timestamp'], ascending=[True, True], inplace=True)

    # Sort by user_id and timestamp
    df.sort_values(by=['user_id', 'timestamp'], ascending=[True, True], inplace=True)

    

    tqdm.pandas(desc="Splitting data")
    splits = df.groupby('user_id').progress_apply(lambda x: split_user_data(x, test_size))

    # Concatenate the results
    train_data = pd.concat([split[0] for split in splits])
    test_data = pd.concat([split[1] for split in splits])
    other_data = pd.concat([split[2] for split in splits])

    # Remove other_data from data by checking user_id and item_id
    data = pd.concat([train_data, test_data])
    # Reset index
    data.reset_index(drop=True, inplace=True)


    # Find number of articles in test_data but not in train_data
    unique_test_articles = set(test_data['item_id'])
    unique_train_articles = set(train_data['item_id'])
    unique_articles_in_test_not_train = unique_test_articles - unique_train_articles
    num_unique_articles_in_test_not_train = len(unique_articles_in_test_not_train)

    # Calculate number of unique user_ids in test_data but not in train_data
    unique_test_user_ids = set(test_data['user_id'])
    unique_train_user_ids = set(train_data['user_id'])
    unique_users_in_test_not_train = unique_test_user_ids - unique_train_user_ids
    num_unique_users_in_test_not_train = len(unique_users_in_test_not_train)
    num_unique_users_in_other_data = len(other_data['user_id'].unique()) if not other_data.empty else 0
    num_unique_articles_in_other_data_not_train = len(other_data[~ other_data['item_id'].isin(unique_train_articles)]['item_id'].unique()) if not other_data.empty else 0

    print(f"Number of articles in test_data but not in train_data: {num_unique_articles_in_test_not_train}")
    print(f"Number of user_ids in test_data but not in train_data: {num_unique_users_in_test_not_train}")
    print(f"Number of user_ids in other_data: {num_unique_users_in_other_data}")
    print(f"Number of articles in other_data but not in train_data: {num_unique_articles_in_other_data_not_train}")

    # Calculate size of test data compared to train data
    size_test_data = len(test_data)
    size_data = len(train_data) + len(test_data)
    percentage_test_data = (size_test_data / size_data) * 100

    print(f"Size of test data: {percentage_test_data:.2f}%")
    print(f"Size of other data: {len(other_data) / size_data * 100:.2f}%")
    print(f"Size of all test data: {(size_test_data + len(other_data)) / size_data * 100:.2f}%")
    return train_data, test_data, other_data

In [ ]:
def map_ids(data, train_data, test_data):
    # Create mapping for userIds and itemIds based on train_df
    unique_user_ids = train_data['user_id'].unique()
    unique_item_ids = train_data['item_id'].unique()

    user_id_mapping = {userId: i for i, userId in enumerate(unique_user_ids)}
    item_id_mapping = {itemId: i for i, itemId in enumerate(unique_item_ids)}

    # Map userIds and itemIds in train_df to new consecutive IDs
    train_data['mapped_user_id'] = train_data['user_id'].map(user_id_mapping)
    train_data['mapped_item_id'] = train_data['item_id'].map(item_id_mapping)

    # Function to map new IDs in test_df and update the mapping accordingly
    def map_and_update_id(id_value, current_mapping):
        if id_value not in current_mapping:
            current_mapping[id_value] = max(current_mapping.values()) + 1
        return current_mapping[id_value]

    test_data['mapped_user_id'] = test_data['user_id'].apply(lambda x: map_and_update_id(x, user_id_mapping))
    test_data['mapped_item_id'] = test_data['item_id'].apply(lambda x: map_and_update_id(x, item_id_mapping))

    if data is not None:
        data['mapped_user_id'] = data['user_id'].map(user_id_mapping).astype('Int64')
        data['mapped_item_id'] = data['item_id'].map(item_id_mapping).astype('Int64')
    else:
        data = pd.concat([train_data, test_data])
    
    data.reset_index(drop=True, inplace=True)
    train_data.reset_index(drop=True, inplace=True)
    test_data.reset_index(drop=True, inplace=True)

    return data, train_data, test_data, user_id_mapping, item_id_mapping

In [ ]:
def load_ml_100k():
    data = pd.read_csv("../results/recommendation/ml-100k/u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])

    data = data[data['rating'] >= 4]

    # Substract 3 to all the ratings
    data['rating'] = data['rating'] - 3

    train_data, test_data, other_data = train_test_split(data)
    print(f"ml-100k - Proportion of positive ratings affected to test set: {round(len(test_data)/(len(test_data)+len(train_data))*100,2)} %")
    
    data, train_data, test_data, user_id_mapping, item_id_mapping = map_ids(data, train_data, test_data)

    # Compute the number of unique users and items and print
    num_unique_users = data['mapped_user_id'].nunique()
    num_unique_items = data['mapped_item_id'].nunique()
    print(f"Number of unique users: {num_unique_users}")
    print(f"Number of unique items: {num_unique_items}")
    # Print the number of interactions
    print(f"Number of interactions: {len(data)}")
    # Compute sparsity
    sparsity = 1 - len(data) / (num_unique_users * num_unique_items)
    print(f"Sparsity: {sparsity:.2%}")
    ######

    ##### Load embeddings #####

    items_embeddings = None
    if ada or bert or content_based:
        items_embeddings = {}
        
        embeddings_file = f"../results/recommendation/embeddings_ml-100k.csv.gz"

        embedding_models = {"ada": ada, "bert": bert}
        if content_based:
            embedding_models = {"ada": True, "bert": True}

        df = pd.read_csv(embeddings_file)
        # Rename movie id to item_id    
        df = df.rename(columns={'movie id': 'item_id'})
                # Iterate over the dictionary
        for embedding_model, is_enabled in embedding_models.items():
            if not is_enabled:
                continue
            # KNN uses the embeddings that were generated
            if content_based and embedding_model + "_embedding" not in df:
                continue
            embedding_key = embedding_model + "_embedding"
            assert train_data[train_data['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty 
            assert test_data[test_data['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty 
            df2 = df[~ df[embedding_key].isnull()]
            df2 = df2.rename(columns={embedding_key: 'embedding'})
            df2['embedding'] = df2['embedding'].apply(ast.literal_eval) #Convert list stored as a str to real list type
            df2['mapped_item_id'] = df2['item_id'].map(item_id_mapping).astype('Int64')
            df2 = df2.dropna(subset=['mapped_item_id']) # If NA, item is not in train set nor test set
            items_embeddings[embedding_model] = df2[['item_id', 'mapped_item_id', 'embedding']]


    ###########################    
    
    return {
        'data': data,
        'train_data': train_data,
        'test_data': test_data,
        'user_id_mapping': user_id_mapping,
        'item_id_mapping': item_id_mapping,
        'items_embeddings': items_embeddings
    }

In [ ]:
def load_LingoRank(strategy: int = 2):
    data_full = pd.read_csv(f"../results/recommendation/Zeegu/strategy{strategy}.csv")
    
    # Rename data_full.article_id as data_full.item_id
    data_full.rename(columns={"article_id": "item_id"}, inplace=True)
    ## Remove the articles for which there is no positive rating 
    # Before removing articles, count the unique articles
    original_unique_articles = data_full['item_id'].nunique()

    # Identify articles that have maximum rating <= 0
    articles_to_remove = data_full.groupby('item_id')['rating'].max()
    articles_to_remove = articles_to_remove[articles_to_remove <= 0].index.tolist()
    articles_to_remove.append(2223234) # Cannot use it for content-based because content is wrong. So we don't consider it at all

    # Remove these articles from data_full
    data_full = data_full[~data_full['item_id'].isin(articles_to_remove)]

    data = data_full[(data_full['rating'] != 0)].copy()
    
    train_data, test_data, other_data = train_test_split(data)
    print(f"Strategy {strategy} - Proportion of positive ratings affected to test set: {round(len(test_data)/(len(test_data)+len(train_data))*100,2)} %")

    data, train_data, test_data, user_id_mapping, item_id_mapping = map_ids(data, train_data, test_data)

    # Compute the number of unique users and items and print
    num_unique_users = data['mapped_user_id'].nunique()
    num_unique_items = data['mapped_item_id'].nunique()
    print(f"Number of unique users: {num_unique_users}")
    print(f"Number of unique items: {num_unique_items}")
    # Print the number of interactions
    print(f"Number of interactions: {len(data)}")
    # Compute sparsity
    sparsity = 1 - len(data) / (num_unique_users * num_unique_items)
    print(f"Sparsity: {sparsity:.2%}")


    ##### Load embeddings #####

    items_embeddings = None
    if ada or bert or content_based:
        items_embeddings = {}
        embeddings_file = f"../results/recommendation/embeddings_strategy{strategy}.csv.gz"

        embedding_models = {"ada": ada, "bert": bert}
        if content_based:
            embedding_models = {"ada": True, "bert": True}

        df = pd.read_csv(embeddings_file)
        df = df.rename(columns={'id': 'item_id'})
        # Iterate over the dictionary
        for embedding_model, is_enabled in embedding_models.items():
            if not is_enabled:
                continue
            # KNN uses the embeddings that were generated
            if content_based and embedding_model + "_embedding" not in df:
                continue

            embedding_key = embedding_model + "_embedding"
            # Remove rows for which ada_embedding is null
            assert data[data['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty # Already removed the article
            assert data_full[data_full['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty # Already removed the article
            
            df2 = df[~ df[embedding_key].isnull()]
            df2 = df2.rename(columns={embedding_key: 'embedding'})
            df2['embedding'] = df2['embedding'].apply(ast.literal_eval) #Convert list stored as a str to real list type
            df2['mapped_item_id'] = df2['item_id'].map(item_id_mapping).astype('Int64')
            df2 = df2.dropna(subset=['mapped_item_id']) # If NA, item is not in train set nor test set
            items_embeddings[embedding_model] = df2[['item_id', 'mapped_item_id', 'embedding']]

    ###########################

    return {
            'data': data,
            'train_data': train_data,
            'test_data': test_data,
            'user_id_mapping': user_id_mapping,
            'item_id_mapping': item_id_mapping,
            'items_embeddings': items_embeddings
        }

In [ ]:
def load_goodreads(subset: str = "children"):

    file_path = f'../results/recommendation/Goodreads/goodreads_reviews_{subset}.json.gz'
    
    # Load the data
    data = pd.read_json(file_path, lines=True, compression='gzip')

    print("JSON LOADED")

    # Select and rename columns
    data = data[['user_id', 'book_id', 'rating', 'date_added']]
    data.rename(columns={'book_id': 'item_id', 'date_added': 'timestamp'}, inplace=True)

    print("COLUMNS SELECTED AND RENAMED")

    data = data[data['rating'] >= 4]

    # Substract 3 to each rating
    data['rating'] = data['rating'] - 3

    print("RATINGS FILTERED")

    # Convert 'timestamp' column to datetime
    data['timestamp'] = pd.to_datetime(data['timestamp'])

    print("TIMESTAMP CONVERTED")

    unique_user_ids = data['user_id'].unique()
    print(f"Number of users: {len(unique_user_ids)}")
    unique_item_ids = data['item_id'].unique()
    print(f"Number of items: {len(unique_item_ids)}")

    train_data, test_data, other_data = train_test_split(data, test_size=0.2, one_core=True)
    print(f"Goodreads {subset} - Proportion of positive ratings affected to test set: {round(len(test_data)/(len(test_data)+len(train_data))*100,2)} %")
    data, train_data, test_data, user_id_mapping, item_id_mapping = map_ids(data, train_data, test_data)

    # Compute the number of unique users and items and print
    num_unique_users = data['mapped_user_id'].nunique()
    num_unique_items = data['mapped_item_id'].nunique()
    print(f"Number of unique users: {num_unique_users}")
    print(f"Number of unique items: {num_unique_items}")
    # Print the number of interactions
    print(f"Number of interactions: {len(data)}")
    # Compute sparsity
    sparsity = 1 - len(data) / (num_unique_users * num_unique_items)
    print(f"Sparsity: {sparsity:.2%}")

    ##### Load embeddings #####

    items_embeddings = None
    if ada or bert or content_based:
        items_embeddings = {}
        
        embeddings_file = f"../results/recommendation/embeddings_goodreads_{subset}.csv.gz"

        embedding_models = {"ada": ada, "bert": bert}

        if content_based:
            embedding_models = {"ada": True, "bert": True}

        df = pd.read_csv(embeddings_file)
        df = df.rename(columns={'book_id': 'item_id'})
        
        # Iterate over the dictionary
        for embedding_model, is_enabled in embedding_models.items():
            if not is_enabled:
                continue
            # KNN uses the embeddings that were generated
            if content_based and embedding_model + "_embedding" not in df:
                continue
            embedding_key = embedding_model + "_embedding"
            assert data[data['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty # Already removed the article
            df2 = df[~ df[embedding_key].isnull()]
            df2 = df2.rename(columns={embedding_key: 'embedding'})
            df2['embedding'] = df2['embedding'].apply(ast.literal_eval) #Convert list stored as a str to real list type
            df2['mapped_item_id'] = df2['item_id'].map(item_id_mapping).astype('Int64')
            df2 = df2.dropna(subset=['mapped_item_id']) # If NA, item is not in train set nor test set
            items_embeddings[embedding_model] = df2[['item_id', 'mapped_item_id', 'embedding']]

    ###########################    
    
    return {
        'data': data,
        'train_data': train_data,
        'test_data': test_data,
        'user_id_mapping': user_id_mapping,
        'item_id_mapping': item_id_mapping,
        'items_embeddings': items_embeddings
    }

In [ ]:
def sparsify_data(data, n_items_per_user):
    # Ensure n_items_per_user is at least 1
    n_items_per_user = max(1, n_items_per_user)

    # Sort data by user and timestamp in descending order (newest first)
    sorted_data = data.sort_values(by=['user_id', 'timestamp'], ascending=[True, False])

    # Group by user and keep only the newest n_items_per_user items for each user
    sparsified_data = sorted_data.groupby('user_id').head(n_items_per_user)

    # Compute the number of unique users and items
    num_unique_users = sparsified_data['user_id'].nunique()
    num_unique_items = sparsified_data['item_id'].nunique()

    # Compute sparsity
    sparsity = 1 - len(sparsified_data) / (num_unique_users * num_unique_items)
    print(f"Number of unique users: {num_unique_users}")
    print(f"Number of unique items: {num_unique_items}")
    print(f"Number of interactions: {len(sparsified_data)}")
    print(f"Sparsity: {sparsity:.6%}")

    return sparsified_data, sparsity

In [ ]:
def load_tomplay():
    file_path = f'../results/recommendation/Tomplay/interactions.csv'
    data = pd.read_csv(file_path, usecols=["USER_ID", "ITEM_ID", "TIMESTAMP"])
    # Rename the columns to lowercase
    data.rename(columns=lambda x: x.lower(), inplace=True)

    data['rating'] = 1

    # Sort the DataFrame by timestamp in descending order
    data = data.sort_values(by='timestamp', ascending=False)

    # Remove duplicates, keeping the first occurrence
    data = data.drop_duplicates(subset=['user_id', 'item_id'], keep='first')

    data, _ = sparsify_data(data, 30)

    unique_user_ids = data['user_id'].unique()
    print(f"Number of users: {len(unique_user_ids)}")
    unique_item_ids = data['item_id'].unique()
    print(f"Number of items: {len(unique_item_ids)}")

    train_data, test_data, other_data = train_test_split(data)
    print(f"Tomplay - Proportion of positive ratings affected to test set: {round(len(test_data)/(len(test_data)+len(train_data))*100,2)} %")

    data, train_data, test_data, user_id_mapping, item_id_mapping = map_ids(data, train_data, test_data)

    # Compute the number of unique users and items and print
    num_unique_users = data['mapped_user_id'].nunique()
    num_unique_items = data['mapped_item_id'].nunique()
    print(f"Number of unique users: {num_unique_users}")
    print(f"Number of unique items: {num_unique_items}")
    # Print the number of interactions
    print(f"Number of interactions: {len(data)}")
    # Compute sparsity
    sparsity = 1 - len(data) / (num_unique_users * num_unique_items)
    print(f"Sparsity: {sparsity:.2%}")

    ##### Load embeddings #####

    items_embeddings = None
    if ada or bert or content_based:
        items_embeddings = {}
        
        embeddings_file = f"../results/recommendation/embeddings_tomplay.csv.gz"

        embedding_models = {"ada": ada, "bert": bert}
        if content_based:
            embedding_models = {"ada": True, "bert": True}

        df = pd.read_csv(embeddings_file)
        df = df.rename(columns={'ITEM_ID': 'item_id'})
        
        # Iterate over the dictionary
        for embedding_model, is_enabled in embedding_models.items():
            if not is_enabled:
                continue
            # KNN uses the embeddings that were generated
            if content_based and embedding_model + "_embedding" not in df:
                continue
            embedding_key = embedding_model + "_embedding"
            assert data[data['item_id'].isin(df[df[embedding_key].isnull()]['item_id'])].empty # Already removed the article
            df2 = df[~ df[embedding_key].isnull()]
            df2 = df2.rename(columns={embedding_key: 'embedding'})
            df2['embedding'] = df2['embedding'].apply(ast.literal_eval) #Convert list stored as a str to real list type
            df2['mapped_item_id'] = df2['item_id'].map(item_id_mapping).astype('Int64')
            df2 = df2.dropna(subset=['mapped_item_id']) # If NA, item is not in train set nor test set
            items_embeddings[embedding_model] = df2[['item_id', 'mapped_item_id', 'embedding']]

    ###########################    


    return {
            'data': data,
            'train_data': train_data,
            'test_data': test_data,
            'user_id_mapping': user_id_mapping,
            'item_id_mapping': item_id_mapping,
            'items_embeddings': items_embeddings
        }

`load_datasets` loads and preprocesses the datasets selected by the configuration. It is deterministic but slow when embeddings are loaded, because they are stored as text and parsed for every item (up to about an hour for Goodreads with both embeddings, about 2 minutes without embeddings), so its output is cached once per combination of dataset and loaded embeddings.

In [ ]:
def dataset_cache_path(config):
    datasets = [flag for flag in DATASET_FLAGS if config.get(flag)]
    # Embeddings loaded by the load_* functions (the Xavier variants load none)
    embeddings = [flag for flag in ["ada", "bert", "content_based"] if config.get(flag)]
    return CACHE_ROOT / f"{'_'.join(datasets)}__{'_'.join(embeddings) or 'none'}.pkl"


def load_datasets(config):
    cache = dataset_cache_path(config)
    # The cache is outdated if embeddings were generated after it
    embeddings_time = max((f.stat().st_mtime for f in DATA.glob("embeddings_*.csv.gz")), default=0)
    if cache.exists() and cache.stat().st_mtime > embeddings_time:
        with open(cache, "rb") as f:
            return pickle.load(f)

    datasets = {}

    # 1. Zeegu
    if lingorank:
        datasets['LingoRank'] = {}
        datasets['LingoRank']['implicit'] = load_LingoRank()
        assert set(datasets['LingoRank']['implicit']['data'].dropna(subset=['mapped_user_id'])['mapped_user_id'].astype(int)) == set(range(int(datasets['LingoRank']['implicit']['data']['mapped_user_id'].max()) + 1)), "User IDs are not continuous and ordered."
        assert set(datasets['LingoRank']['implicit']['data'].dropna(subset=['mapped_item_id'])['mapped_item_id'].astype(int)) == set(range(int(datasets['LingoRank']['implicit']['data']['mapped_item_id'].max()) + 1)), "Item IDs are not continuous and ordered."

    # 2. MovieLens-100k
    if ml_100k:
        datasets['ml-100k'] = {}
        datasets['ml-100k']['implicit'] = load_ml_100k()
        assert set(datasets['ml-100k']['implicit']['data'].dropna(subset=['mapped_user_id'])['mapped_user_id'].astype(int)) == set(range(int(datasets['ml-100k']['implicit']['data']['mapped_user_id'].max()) + 1)), "User IDs are not continuous and ordered."
        assert set(datasets['ml-100k']['implicit']['data'].dropna(subset=['mapped_item_id'])['mapped_item_id'].astype(int)) == set(range(int(datasets['ml-100k']['implicit']['data']['mapped_item_id'].max()) + 1)), "Item IDs are not continuous and ordered."

    # 3. Goodreads
    if goodreads:
        datasets['Goodreads'] = {}
        datasets['Goodreads']['implicit'] = load_goodreads()
        assert set(datasets['Goodreads']['implicit']['data'].dropna(subset=['mapped_user_id'])['mapped_user_id'].astype(int)) == set(range(int(datasets['Goodreads']['implicit']['data']['mapped_user_id'].max()) + 1)), "User IDs are not continuous and ordered."
        assert set(datasets['Goodreads']['implicit']['data'].dropna(subset=['mapped_item_id'])['mapped_item_id'].astype(int)) == set(range(int(datasets['Goodreads']['implicit']['data']['mapped_item_id'].max()) + 1)), "Item IDs are not continuous and ordered."

    # 4. Tomplay
    if tomplay:
        datasets['Tomplay'] = {}
        datasets['Tomplay']['implicit'] = {}
        datasets['Tomplay']['implicit'] = load_tomplay()
        assert set(datasets['Tomplay']['implicit']['data'].dropna(subset=['mapped_user_id'])['mapped_user_id'].astype(int)) == set(range(int(datasets['Tomplay']['implicit']['data']['mapped_user_id'].max()) + 1)), "User IDs are not continuous and ordered."
        assert set(datasets['Tomplay']['implicit']['data'].dropna(subset=['mapped_item_id'])['mapped_item_id'].astype(int)) == set(range(int(datasets['Tomplay']['implicit']['data']['mapped_item_id'].max()) + 1)), "Item IDs are not continuous and ordered."

    # Written to a temporary file first: several runs may compute the same cache
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_suffix(f".tmp{os.getpid()}")
    with open(tmp, "wb") as f:
        pickle.dump(datasets, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, cache)
    return datasets

## Evaluation metrics

In [ ]:
def ndcg_at_k(recommendations, real_items, k):
    """
    Compute NDCG at rank k
    """
    ranked_list = [1 if item_id in real_items else 0 for item_id in recommendations]
    num_relevant = min(k, len(real_items))
    ideal_list = [1] * num_relevant + [0] * (k - num_relevant)
    dcg = sum((2 ** rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(ranked_list[:k]))
    idcg = sum((2 ** rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(ideal_list[:k]))
    return dcg / idcg if idcg > 0 else 0

def mrr_at_k(recommendations, real_items, k):
    """
    Compute MRR at rank k
    """
    ranked_list = [1 if item_id in real_items else 0 for item_id in recommendations]
    for idx, rel in enumerate(ranked_list[:k]):
        if rel > 0:
            return 1 / (idx + 1)
    return 0

def precision_at_k(recommendations, real_items, k):
    """
    Compute Precision at rank k
    """
    return sum([1 if item_id in real_items else 0 for item_id in recommendations[:k]]) / min(k, len(recommendations))

def recall_at_k(recommendations, real_items, k):
    """
    Compute Recall at rank k
    """
    return sum([1 if item_id in real_items else 0 for item_id in recommendations[:k]]) / len(real_items)

def f1_at_k(precision_at_k, recall_at_k):
    """
    Compute F1 at rank k
    """
    if precision_at_k == 0 and recall_at_k == 0:
        return 0

    f1_score = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k)
    return f1_score

def average_precision_at_k(recommendations, test_items, k):

    recommendations_presence = [1 if item_id in test_items else 0 for item_id in recommendations[:k]]

    # Keep track of the number of relevant items found
    num_relevant = 0
    # Keep track of the sum of precisions
    sum_precisions = 0
    # We are interested in the precision at each point a relevant document is retrieved
    for idx in range(min(k, len(recommendations_presence))):
        # Check if the item is relevant
        if recommendations_presence[idx] == 1:
            # Increment the count of relevant items
            num_relevant += 1
            # Update the sum of precisions
            sum_precisions += num_relevant / (idx + 1)

    return sum_precisions / min(k, len(test_items))

In [ ]:
def evaluate(test_data, data_full, data_strategy, RS_strategy_name, RS_strategy, model_name, model, dataset, k=5, progress=True):

    global user_embedding
    global item_embeddings
    
    ndcgs = []
    rrs = []
    aps = []  
    precisions = []
    recalls = []
    f1s = []
    all_recommendations = []
    all_really_in_test = []

    # Loop for each user in test_df
    user_ids = test_data['mapped_user_id'].unique()

    if progress:
        user_ids_for_loop = tqdm(user_ids, desc="Users")
    else:
        user_ids_for_loop = user_ids   
    for user in user_ids_for_loop:

        if RS_strategy_name=="CF" and data_strategy=="implicit":
            top_k_recommendations = model['model'].recommend(user, RS_strategy['user_item_train_data'][user], k)[0]
            user_test_items = test_data[test_data['mapped_user_id'] == user].mapped_item_id.tolist()
        
        elif RS_strategy_name=="content-based":

            # Get the list of item_ids from train_data that the user has already interacted with.
            train_data_items = data_strategy['train_data'][data_strategy['train_data']['mapped_user_id'] == user]['mapped_item_id'].tolist()
            # Get the similarity scores and filter out items present in train_data.
            filtered_sim_scores = model['sim_matrix'].loc[user].drop(train_data_items)
            # Now sort the remaining items and select the top k items.
            top_k_recommendations = np.array(filtered_sim_scores.sort_values(ascending=False).index.tolist()[:k])
            user_test_items = test_data[test_data['mapped_user_id'] == user].mapped_item_id.tolist()

        elif RS_strategy_name=="graph-based":

            top_k_recommendations = model['model'].recommend_k_items(pd.DataFrame({'userID': [user]}), top_k=k, remove_seen=True)
            top_k_recommendations = top_k_recommendations['itemID'].tolist()
            user_test_items = test_data[test_data['mapped_user_id'] == user].mapped_item_id.tolist()

        ndcg = ndcg_at_k(top_k_recommendations, user_test_items, k)
        rr = mrr_at_k(top_k_recommendations, user_test_items, k)
        ap = average_precision_at_k(top_k_recommendations, user_test_items, k)
        precision = precision_at_k(top_k_recommendations, user_test_items, k)
        recall = recall_at_k(top_k_recommendations, user_test_items, k)
        f1 = f1_at_k(precision, recall)

        ndcgs.append(ndcg)
        rrs.append(rr)
        aps.append(ap)
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    mndcg = np.mean(ndcgs)
    mrr = np.mean(rrs)
    map = np.mean(aps)
    mprecision = np.mean(precisions)
    mrecall = np.mean(recalls)
    mf1 = np.mean(f1s)

    return {
        'mndcg': mndcg,
        'mrr': mrr,
        'map': map,
        'mrecall': mrecall,
        'mprecision': mprecision,
        'mf1': mf1,
        'ndcgs': ndcgs,
        'rrs': rrs,
        'aps': aps,
        'recalls': recalls,
        'precisions': precisions,
        'f1s': f1s,
        'all_recommendations': all_recommendations,
        'all_really_in_test': all_really_in_test
    }

## Models

### User embeddings and KNN

The embedding of a user is the mean of the embeddings of their `n` most recent training items. `compute_user_embeddings` gets the rows of these items once from a dictionary, because selecting them with a filter on `embeddings_df` for every user is quadratic.

KNN results can differ slightly with regenerated embeddings: some Zeeguu articles have identical texts, and ties are broken by the order of the rows of the embedding file.

In [ ]:
def compute_user_embeddings(train_data, embeddings_df, n):
    # Last n items of each user, in increasing mapped_user_id order
    train_data_sorted = train_data.sort_values(by=['mapped_user_id', 'timestamp'], ascending=[True, False])
    last_items = train_data_sorted.groupby('mapped_user_id')['mapped_item_id'].apply(lambda s: s.head(n).tolist())
    item_ids = embeddings_df['mapped_item_id'].to_numpy()
    row_of_item = {}
    for row, item in enumerate(item_ids):
        row_of_item.setdefault(item, []).append(row)
    matrix = np.stack(embeddings_df['embedding'].to_numpy())
    users, embeddings, num_items, real_nb_neigh = [], [], [], []
    for user_id, items in last_items.items():
        real_nb_neigh.append(len(items))
        rows = sorted(r for item in set(items) for r in row_of_item.get(item, []))
        emb = matrix[rows]
        users.append(user_id)
        embeddings.append(np.mean(emb, axis=0))
        num_items.append(len(emb))
    user_embeddings = pd.DataFrame({'mapped_user_id': users, 'embedding': embeddings, 'num_items': num_items})
    return user_embeddings, sum(real_nb_neigh) / len(real_nb_neigh)


def compute_similarity_matrix(items_embeddings, users_embeddings, user=None):

    if user is not None:
        users_embeddings = users_embeddings[users_embeddings['mapped_user_id'] == user]

    # Convert the embeddings to numpy arrays
    item_embeddings_array = np.vstack(items_embeddings['embedding'].apply(np.array))
    user_embeddings_array = np.vstack(users_embeddings['embedding'].apply(np.array))

    # Compute the cosine similarity matrix
    cosine_sim_matrix = cosine_similarity(user_embeddings_array, item_embeddings_array)

    # Convert the cosine similarity matrix into a DataFrame
    # Use user_ids for the index and item_ids for the columns
    cosine_sim_df = pd.DataFrame(cosine_sim_matrix,
                                index=users_embeddings['mapped_user_id'],
                                columns=items_embeddings['mapped_item_id'])
    
    return cosine_sim_df

In [ ]:
def build_knn(datasets):
    for key_dataset, dataset in tqdm(datasets.items()):
        dataset['implicit']['RS_strategy']['content-based'] = {}
        dataset['implicit']['RS_strategy']['content-based']['models'] = {}
        assert dataset['implicit']['items_embeddings'], f"No embeddings for {key_dataset}: see b_GenerateEmbeddings.ipynb"
        for embedding_model in dataset['implicit']['items_embeddings']:
            dataset['implicit']['RS_strategy']['content-based']['models'][embedding_model] = {}
            dataset['implicit']['RS_strategy']['content-based']['models'][embedding_model]['k'] = {n: {} for n in KNN_PROFILE_SIZES}

            train_data = dataset['implicit']['train_data']
            
            cb_embed = dataset['implicit']['RS_strategy']['content-based']['models'][embedding_model]
            for key in cb_embed['k']:
                cb_embed['k'][key]['users_embeddings'], cb_embed['k'][key]['mean_k'] = compute_user_embeddings(train_data, dataset['implicit']['items_embeddings'][embedding_model], key)
                # dataset['implicit']['RS_strategy']['content-based']['models'][embedding_model]['k'][key]['sim_matrix'] = compute_similarity_matrix(dataset['implicit']['items_embeddings'][embedding_model], cb_embed['k'][key]['users_embeddings'])

### LightGCN

The class of `recommenders` is adapted in two ways:

* `new_init` and `new_init_weights` initialise the user and item embeddings with the LLM embeddings when `initial_embeddings` is given, and with the default Xavier initialisation otherwise.
* `fast_recommend_k_items` replaces `LightGCN.recommend_k_items` and gives the same output, faster: the training matrix is converted to CSR once instead of at every call (which took about 4 seconds per user on Goodreads), and the top-k items are selected on each batch of users instead of on the dense users × items score matrix.

In [ ]:
def new_init(self, hparams, data, initial_embeddings=None ,seed=None):
    
    """Initializing the model. Create parameters, placeholders, embeddings and loss function.

    Args:
        hparams (HParams): A HParams object, hold the entire set of hyperparameters.
        data (object): A recommenders.models.deeprec.DataModel.ImplicitCF object, load and process data.
        seed (int): Seed.

    """

    tf.compat.v1.set_random_seed(seed)
    np.random.seed(seed)

    self.data = data
    self.epochs = hparams.epochs
    self.lr = hparams.learning_rate
    self.emb_dim = hparams.embed_size
    self.batch_size = hparams.batch_size
    self.n_layers = hparams.n_layers
    self.decay = hparams.decay
    self.eval_epoch = hparams.eval_epoch
    self.top_k = hparams.top_k
    self.save_model = hparams.save_model
    self.save_epoch = hparams.save_epoch
    self.metrics = hparams.metrics
    self.model_dir = hparams.MODEL_DIR
    self.initial_embeddings = initial_embeddings

    metric_options = ["map", "ndcg", "precision", "recall"]
    for metric in self.metrics:
        if metric not in metric_options:
            raise ValueError(
                "Wrong metric(s), please select one of this list: {}".format(
                    metric_options
                )
            )

    self.norm_adj = data.get_norm_adj_mat()

    self.n_users = data.n_users
    self.n_items = data.n_items

    self.users = tf.compat.v1.placeholder(tf.int32, shape=(None,))
    self.pos_items = tf.compat.v1.placeholder(tf.int32, shape=(None,))
    self.neg_items = tf.compat.v1.placeholder(tf.int32, shape=(None,))

    self.weights = self._init_weights()
    self.ua_embeddings, self.ia_embeddings = self._create_lightgcn_embed()

    self.u_g_embeddings = tf.nn.embedding_lookup(
        params=self.ua_embeddings, ids=self.users
    )
    self.pos_i_g_embeddings = tf.nn.embedding_lookup(
        params=self.ia_embeddings, ids=self.pos_items
    )
    self.neg_i_g_embeddings = tf.nn.embedding_lookup(
        params=self.ia_embeddings, ids=self.neg_items
    )
    self.u_g_embeddings_pre = tf.nn.embedding_lookup(
        params=self.weights["user_embedding"], ids=self.users
    )
    self.pos_i_g_embeddings_pre = tf.nn.embedding_lookup(
        params=self.weights["item_embedding"], ids=self.pos_items
    )
    self.neg_i_g_embeddings_pre = tf.nn.embedding_lookup(
        params=self.weights["item_embedding"], ids=self.neg_items
    )

    self.batch_ratings = tf.matmul(
        self.u_g_embeddings,
        self.pos_i_g_embeddings,
        transpose_a=False,
        transpose_b=True,
    )

    self.mf_loss, self.emb_loss = self._create_bpr_loss(
        self.u_g_embeddings, self.pos_i_g_embeddings, self.neg_i_g_embeddings
    )
    self.loss = self.mf_loss + self.emb_loss

    self.opt = tf.compat.v1.train.AdamOptimizer(learning_rate=self.lr).minimize(
        self.loss
    )
    self.saver = tf.compat.v1.train.Saver(max_to_keep=1)

    gpu_options = tf.compat.v1.GPUOptions(allow_growth=True)
    self.sess = tf.compat.v1.Session(
        config=tf.compat.v1.ConfigProto(gpu_options=gpu_options)
    )
    self.sess.run(tf.compat.v1.global_variables_initializer())

In [ ]:
def new_init_weights(self):
    """Initialize user and item embeddings.

    Returns:
        dict: With keys `user_embedding` and `item_embedding`, embeddings of all users and items.

    """
    all_weights = dict()
    initializer = tf.compat.v1.keras.initializers.VarianceScaling(
        scale=1.0, mode="fan_avg", distribution="uniform"
    )

    if self.initial_embeddings is not None:
        all_weights["item_embedding"] = tf.Variable(self.initial_embeddings['item_embedding'], dtype=tf.float32, name="item_embedding")
        all_weights["user_embedding"] = tf.Variable(self.initial_embeddings['user_embedding'], dtype=tf.float32, name="user_embedding")
        print("Using pretrained embeddings.")
    else:
        all_weights["user_embedding"] = tf.Variable(
            initializer([self.n_users, self.emb_dim]), name="user_embedding"
        )
        all_weights["item_embedding"] = tf.Variable(
            initializer([self.n_items, self.emb_dim]), name="item_embedding"
        )
        print("Using xavier initialization.")

    return all_weights

In [ ]:
def fast_recommend_k_items(self, test, top_k=10, sort_top_k=True, remove_seen=True, use_id=False):
    data = self.data
    if not use_id:
        user_ids = np.array([data.user2id[x] for x in test[data.col_user].unique()])
    else:
        user_ids = np.array(test[data.col_user].unique())
    if any(np.isnan(user_ids)):
        raise ValueError("LightGCN cannot score users that are not in the training set")
    if remove_seen:
        if getattr(data, "_R_csr", None) is None:
            data._R_csr = data.R.tocsr()
        R_csr = data._R_csr
    u_batch_size = self.batch_size
    n_user_batchs = len(user_ids) // u_batch_size + 1
    top_items, top_scores = [], []
    for u_batch_id in range(n_user_batchs):
        start = u_batch_id * u_batch_size
        end = (u_batch_id + 1) * u_batch_size
        user_batch = user_ids[start:end]
        item_batch = range(self.data.n_items)
        rate_batch = self.sess.run(
            self.batch_ratings, {self.users: user_batch, self.pos_items: item_batch}
        )
        batch_scores = np.array(rate_batch)
        if len(batch_scores) == 0:
            continue
        if remove_seen:
            batch_scores += R_csr[user_batch, :] * -np.inf
        items, scores = get_top_k_scored_items(scores=batch_scores, top_k=top_k, sort_top_k=sort_top_k)
        top_items.append(items)
        top_scores.append(scores)
    top_items = np.concatenate(top_items, axis=0)
    top_scores = np.concatenate(top_scores, axis=0)

    df = pd.DataFrame(
        {
            data.col_user: np.repeat(
                test[data.col_user].drop_duplicates().values, top_items.shape[1]
            ),
            data.col_item: top_items.flatten()
            if use_id
            else [data.id2item[item] for item in top_items.flatten()],
            data.col_prediction: top_scores.flatten(),
        }
    )

    return df.replace(-np.inf, np.nan).dropna()

Grid search of LightGCN: for each hyperparameter combination, training with early stopping on NDCG@5 (patience of 10 epochs). The best checkpoint over all the combinations is kept. Each run has its own checkpoint folder, so that parallel runs do not overwrite each other's best model.

The best NDCG starts at -1 so that a model that degenerates (NDCG = 0 at every epoch, observed for LMF) is kept and evaluated, instead of leaving no model to evaluate.

In [ ]:
def train_lightgcn(datasets, run_name):
    for key_dataset, dataset in tqdm(datasets.items()):
        dataset['implicit']['RS_strategy']['graph-based'] = {}
        graph_based = dataset['implicit']['RS_strategy']['graph-based']
        dataset['implicit']['RS_strategy']['graph-based']['models'] = {}
        dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN'] = {}

        dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'] = {}

        print("="*100)
        print(key_dataset)
        print("="*100)

        embeddings_models = ["ada" if (ada or xavier_ada) else None] + ["bert" if (bert or xavier_bert) else None]
        embedding_models = [embeddings_model for embeddings_model in embeddings_models if embeddings_model is not None]
        for embedding_model in embedding_models:
            dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'][embedding_model] = {}
            dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'][embedding_model]['method'] = {}
            print("-"*100)
            print(embedding_model)
            print("-"*100)
                
            for method in [method for method in ['Xavier' if xavier_ada or xavier_bert else None] + ['precomputed' if ada or bert else None] if method is not None]:
                print("*"*50)
                print(method)
                print("*"*50)

                dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'][embedding_model]['method'][method] = {}

                train_data = dataset['implicit']['train_data'].copy()
                train_data = train_data.rename(columns = {'mapped_user_id':'userID', 'mapped_item_id':'itemID'})
            
                test_data = dataset['implicit']['test_data'].copy()
                test_data = test_data.rename(columns = {'mapped_user_id':'userID', 'mapped_item_id':'itemID'})
                    
                dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['data'] = ImplicitCF(train=train_data, test=test_data, seed=my_seed, col_user='userID', col_item='itemID', col_rating='rating')
                data = dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['data']

                try:
                    embed_size = len(dataset['implicit']['items_embeddings'][embedding_model]['embedding'].iloc[0])
                except:
                    if embedding_model=="ada":
                        embed_size = 1536
                    elif embedding_model=="bert":
                        embed_size = 768
                assert (xavier_ada or ada) and embed_size==1536 or (xavier_bert or bert) and embed_size==768, "Embedding size is not correct"
                print(f"Embedding size: {embed_size}")

                if method=="precomputed":
                    item_embeddings = dataset['implicit']['items_embeddings'][embedding_model]

                    item_embeddings = item_embeddings[item_embeddings.mapped_item_id.isin(set(train_data.itemID).union(set(test_data.itemID)))]
                    item_embeddings = item_embeddings.sort_values(by='mapped_item_id', ascending=True)
                    assert item_embeddings[item_embeddings.mapped_item_id.isin(set(train_data.itemID).union(set(test_data.itemID)))].mapped_item_id.is_monotonic_increasing, 'not ordered by item_id'

                    initial_embeddings = {}
                    initial_embeddings['item_embedding'] = np.vstack(item_embeddings[item_embeddings.mapped_item_id.isin(set(train_data.itemID).union(test_data.itemID))].embedding.values)
                    ns = lightgcn_params['n']
                elif method=="Xavier":
                    initial_embeddings = None
                    ns = [None]

                all_hparams_best_model = None
                all_hparams_best_ndcg = -1
                best_hparams = None
                best_hparams_for_init = None
                best_initial_embeddings = None

                total_configs = len(lightgcn_params['n_layers'])*len(ns)*len(lightgcn_params['lr'])

                for n_layers, n, lr in tqdm(itertools.product(lightgcn_params['n_layers'], ns , lightgcn_params['lr']), total=total_configs):
                    
                    print("^"*50)    
                    print(f"n_layers: {n_layers} - n: {n} - lr: {lr}")
                    print("^"*50) 
                    if method=="precomputed":
                        user_embeddings, real_nb_neigh = compute_user_embeddings(dataset['implicit']['train_data'], item_embeddings, n)
                        initial_embeddings['user_embedding'] = np.vstack(user_embeddings[user_embeddings.mapped_user_id.isin(set(train_data.userID))].embedding.values)
                

                    hparams = prepare_hparams(                          
                                learning_rate=lr,
                                eval_epoch=10000000,
                                top_k=5,
                                save_model=False,
                                epochs=1,
                                save_epoch=1,
                                model_type="lightgcn",
                                embed_size=embed_size,
                                n_layers=n_layers,
                                batch_size=1024,
                                decay=0.0001,
                                metrics=["recall", "ndcg", "precision", "map"],
                                MODEL_DIR="./tests/resources/deeprec/lightgcn/model/lightgcn_model/"
                                )
                
                
                    best_model = None
                    best_ndcg = 0
                    tf.compat.v1.set_random_seed(my_seed)
                    tf.random.set_seed(my_seed)
                    np.random.seed(my_seed)
                    random.seed(my_seed)
                    model = LightGCN(hparams, data, initial_embeddings=initial_embeddings, seed=my_seed)
                    save_path = str(CHECKPOINT_ROOT / run_name / f"lightgcn_model_{embedding_model}_{method}" / "best_model")
                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                    patience_max = 10

                    with Timer() as train_time:
                        for epoch in range(sys.maxsize):
                            model.fit()
                            eval_start = time.time()
                            recall, ndcg, precision, map = model.run_eval()
                            eval_end = time.time()
                            eval_time = eval_end - eval_start         
                            print(f"Evaluation time: {eval_time:.1f}s")
                            print(f"Epoch {epoch} - Recall@5: {recall:.4f} - NDCG@5: {ndcg:.4f} - Precision@5: {precision:.4f} - MAP@5: {map:.4f}")
                            print("------------------------")
                            if ndcg > all_hparams_best_ndcg:
                                all_hparams_best_ndcg = ndcg
                                best_hparams = {'n_layers': n_layers, 'n': n, 'lr': lr}
                                best_hparams_for_init = hparams
                                best_initial_embeddings = initial_embeddings
                                model.saver.save(model.sess, save_path)
                            if ndcg > best_ndcg:
                                patience = 0
                                best_ndcg = ndcg
                                best_epoch = epoch
                            else:
                                patience += 1
                            
                            if patience == patience_max:
                                print("="*25 + f"Best NDCG: {best_ndcg:.4f} at epoch {best_epoch}" + "="*25)
                                break
                    print("Took {} seconds for training.".format(train_time.interval))
                    
                    model.sess.close()  # Close the existing session
                    tf.compat.v1.Session().close()
                    # Reset the default graph
                    tf.compat.v1.reset_default_graph()


                model = LightGCN(best_hparams_for_init, data, initial_embeddings=best_initial_embeddings, seed=my_seed)
                model.load(save_path)
                dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'][embedding_model]['method'][method]['model'] = model
                dataset['implicit']['RS_strategy']['graph-based']['models']['LightGCN']['embedding_model'][embedding_model]['method'][method]['config'] = best_hparams
                tf.compat.v1.Session().close()
                # Reset the default graph
                tf.compat.v1.reset_default_graph()
                model = None

### Collaborative filtering

Same grid search and early stopping as LightGCN, with the `implicit` library.

In [ ]:
def build_cf_data(datasets):
    # Create user-item sparse matrix
    for key_dataset, dataset in tqdm(datasets.items()):
        dataset['implicit']['RS_strategy']['CF'] = {}
        cf_strategy = dataset['implicit']['RS_strategy']['CF']
        train_data = dataset['implicit']['train_data']

        cf_strategy['user_item_train_data'] = sparse.csr_matrix(
            (train_data['rating'].astype(float), (train_data['mapped_user_id'], train_data['mapped_item_id']))
        )

In [ ]:
def train_cf(datasets):
    for key_dataset, dataset in tqdm(datasets.items(), desc="Datasets"):
        cf_strategy = dataset['implicit']['RS_strategy']['CF']
        cf_strategy['models'] = {}
        if als:
            cf_strategy['models']['ALS'] = {}
            cf_strategy['models']['ALS']['config'] = {}
            
            all_hparams_best_model = None
            all_hparams_best_ndcg = -1
            best_hparams = None
              
            for factors, regularization, alpha in tqdm(itertools.product(als_params['factors'], als_params['regularization'], als_params['alpha']), total=len(list(itertools.product(*als_params.values())))):
                
                model = implicit.als.AlternatingLeastSquares(factors=factors, regularization=regularization, alpha=alpha, iterations=1, calculate_training_loss=True, random_state=my_seed, use_gpu=False)
                training_data = cf_strategy['user_item_train_data'].astype('double')
                test_data = dataset['implicit']['test_data']
                best_ndcg = 0
                patience = 0
                patience_max = 10
                for epoch in range(sys.maxsize):
                    model.fit(training_data, show_progress=False)
                    # Evaluate the model
                    results = evaluate(test_data, None, 'implicit', 'CF', dataset['implicit']['RS_strategy']['CF'], 'ALS', {'model':model}, dataset, k=5, progress=False)
                    ndcg = results['mndcg']

                    if ndcg > all_hparams_best_ndcg:
                        all_hparams_best_ndcg = ndcg
                        best_hparams = {'factors': factors, 'regularization': regularization, 'alpha': alpha}
                        all_hparams_best_model = copy.deepcopy(model)
                    if ndcg > best_ndcg:
                        patience = 0
                        best_ndcg = ndcg
                        best_epoch = epoch
                    else:
                        patience += 1
                    
                    if patience == patience_max:
                        print("-"*25 + f"hparams: {{'factors': {factors}, 'regularization': {regularization}, 'alpha': {alpha}}}" + "-"*25)
                        print("="*25 + f"Best NDCG: {best_ndcg:.4f} at epoch {best_epoch}" + "="*25)
                        break
                
            cf_strategy['models']['ALS']['model'] = all_hparams_best_model
            cf_strategy['models']['ALS']['config'] = best_hparams

        if bpr:
            cf_strategy['models']['BPR'] = {}
            cf_strategy['models']['BPR']['config'] = {}
            
            all_hparams_best_model = None
            all_hparams_best_ndcg = -1
            best_hparams = None

            for factors, regularization, learning_rate, verify_negative_samples in tqdm(itertools.product(bpr_params['factors'], bpr_params['regularization'], bpr_params['learning_rate'], bpr_params['verify_negative_samples']), total=len(list(itertools.product(*bpr_params.values())))):
                
                model = implicit.bpr.BayesianPersonalizedRanking(factors=factors, learning_rate=learning_rate, regularization=regularization, verify_negative_samples=verify_negative_samples, iterations=1, random_state=my_seed, use_gpu=False)
                training_data = cf_strategy['user_item_train_data'].astype('double')
                test_data = dataset['implicit']['test_data']
                best_ndcg = 0
                patience = 0
                patience_max = 10
                for epoch in range(sys.maxsize):
                    model.fit(training_data, show_progress=False)
                    # Evaluate the model
                    results = evaluate(test_data, None, 'implicit', 'CF', dataset['implicit']['RS_strategy']['CF'], 'BPR', {'model':model}, dataset, k=5, progress=False)
                    ndcg = results['mndcg']

                    if ndcg > all_hparams_best_ndcg:
                        all_hparams_best_ndcg = ndcg
                        best_hparams = {'factors': factors, 'regularization': regularization, 'learning_rate': learning_rate, 'verify_negative_samples': verify_negative_samples}
                        all_hparams_best_model = copy.deepcopy(model)
                    if ndcg > best_ndcg:
                        patience = 0
                        best_ndcg = ndcg
                        best_epoch = epoch
                    else:
                        patience += 1
                    
                    if patience == patience_max:
                        print("-"*25 + f"hparams: {{'factors': {factors}, 'regularization': {regularization}, 'learning_rate': {learning_rate}, 'verify_negative_samples': {verify_negative_samples}}}" + "-"*25)
                        print("="*25 + f"Best NDCG: {best_ndcg:.4f} at epoch {best_epoch}" + "="*25)
                        break
                
            cf_strategy['models']['BPR']['model'] = all_hparams_best_model
            cf_strategy['models']['BPR']['config'] = best_hparams
        if lmf:
            
            cf_strategy['models']['LMF'] = {}
            cf_strategy['models']['LMF']['config'] = {}
            
            all_hparams_best_model = None
            all_hparams_best_ndcg = -1
            best_hparams = None

            for factors, regularization, learning_rate, neg_prop in tqdm(itertools.product(lmf_params['factors'], lmf_params['regularization'], lmf_params['learning_rate'], lmf_params['neg_prop']), total=len(list(itertools.product(*lmf_params.values())))):
                
                model = implicit.lmf.LogisticMatrixFactorization(factors=factors, learning_rate=learning_rate, regularization=regularization, neg_prop=neg_prop, iterations=1, random_state=my_seed, use_gpu=False)
                training_data = cf_strategy['user_item_train_data'].astype('double')
                test_data = dataset['implicit']['test_data']
                best_ndcg = 0
                patience = 0
                patience_max = 10
                for epoch in range(sys.maxsize):
                    model.fit(training_data, show_progress=False)
                    # Evaluate the model
                    results = evaluate(test_data, None, 'implicit', 'CF', dataset['implicit']['RS_strategy']['CF'], 'LMF', {'model':model}, dataset, k=5, progress=False)
                    ndcg = results['mndcg']

                    if ndcg > all_hparams_best_ndcg:
                        all_hparams_best_ndcg = ndcg
                        best_hparams = {'factors': factors, 'regularization': regularization, 'learning_rate': learning_rate, 'neg_prop': neg_prop}
                        all_hparams_best_model = copy.deepcopy(model)
                    if ndcg > best_ndcg:
                        patience = 0
                        best_ndcg = ndcg
                        best_epoch = epoch
                    else:
                        patience += 1
                    
                    if patience == patience_max:
                        print("-"*25 + f"hparams: {{'factors': {factors}, 'regularization': {regularization}, 'learning_rate': {learning_rate}, 'neg_prop': {neg_prop}}}" + "-"*25)
                        print("="*25 + f"Best NDCG: {best_ndcg:.4f} at epoch {best_epoch}" + "="*25)
                        break
                
            cf_strategy['models']['LMF']['model'] = all_hparams_best_model
            cf_strategy['models']['LMF']['config'] = best_hparams

## One run

`run` trains and evaluates the models of one configuration and one seed, then saves their metrics. `saved_models` lists the models of a run, with keys `(dataset, model, variant)`, where `variant` is the profile size `k=<n>` of KNN.

In [ ]:
def evaluate_models(datasets):
    for key_dataset, dataset in datasets.items():
        print(key_dataset)
        for key_data_strategy, data_strategy in dataset.items():
            print('\t' + key_data_strategy) 
            for key_RS_strategy, RS_strategy in data_strategy['RS_strategy'].items():
                print('\t\t' + key_RS_strategy)
                for key_model, model in RS_strategy['models'].items():
                    print('\t\t\t' + key_model)
                    if model.get('embedding_model', None):
                        for key_embedding_model, embedding_model in model['embedding_model'].items():
                            for method in embedding_model['method'].keys():
                                print('\t\t\t\t' + key_embedding_model)
                                model['embedding_model'][key_embedding_model]['method'][method]['metrics'] = evaluate(data_strategy['test_data'], data_strategy.get('data_full', None), key_data_strategy, key_RS_strategy, RS_strategy, key_model, model['embedding_model'][key_embedding_model]['method'][method], key_dataset, k=5)
                    elif model.get('k', None):
                        for nb_neigh in model['k'].keys():
                            model['k'][nb_neigh]['metrics'] = {}
                            model['k'][nb_neigh]['sim_matrix'] = compute_similarity_matrix(dataset['implicit']['items_embeddings'][key_model], model['k'][nb_neigh]['users_embeddings'])
                            model['k'][nb_neigh]['metrics'] = evaluate(data_strategy['test_data'], data_strategy.get('data_full', None), data_strategy, key_RS_strategy, RS_strategy, key_model, model['k'][nb_neigh], key_dataset, k=5)
                            del  model['k'][nb_neigh]['sim_matrix']

                    else:
                        model['metrics'] = {}
                        model['metrics'] = evaluate(data_strategy['test_data'], data_strategy.get('data_full', None), key_data_strategy, key_RS_strategy, RS_strategy, key_model, model, key_dataset, k=5)

In [ ]:
PER_USER_KEYS = ["mndcg", "mrr", "map", "mrecall", "mprecision", "mf1", "ndcgs", "rrs", "aps", "recalls", "precisions", "f1s"]


def saved_models(datasets):
    """Result entry of every model of a run."""
    models = {}
    for key_dataset, dataset in datasets.items():
        for rs in dataset["implicit"]["RS_strategy"].values():
            for key_model, model in rs.get("models", {}).items():
                if model.get("k", None):
                    for k, entry in model["k"].items():
                        models[(key_dataset, f"KNN {key_model}", f"k={k}")] = entry
                elif model.get("embedding_model", None):
                    for emb, emb_entry in model["embedding_model"].items():
                        for method, entry in emb_entry["method"].items():
                            name = f"LightGCN {'Xavier ' if method == 'Xavier' else ''}{emb}"
                            models[(key_dataset, name, "")] = entry
                else:
                    models[(key_dataset, key_model, "")] = model
    return models


def per_user_metrics(datasets):
    """Metrics of every model, aggregated and per test user."""
    per_user = {key: {m: entry["metrics"][m] for m in PER_USER_KEYS} for key, entry in saved_models(datasets).items()}
    for key_dataset, dataset in datasets.items():
        test_users = dataset["implicit"]["test_data"]["mapped_user_id"].unique().tolist()
        per_user[(key_dataset, "_test_users", "")] = {"user_ids": test_users}
    return per_user


def evaluation_table(datasets):
    """Metrics @5 and best hyperparameters of every model."""
    rows = []
    for (key_dataset, model, variant), entry in saved_models(datasets).items():
        metrics = entry["metrics"]
        rows.append({"dataset": key_dataset, "model": model, "variant": variant, "best_hparams": entry.get("config"),
                     "Recall@5": metrics["mrecall"], "Precision@5": metrics["mprecision"], "F1@5": metrics["mf1"],
                     "NDCG@5": metrics["mndcg"], "MRR@5": metrics["mrr"], "MAP@5": metrics["map"]})
    return pd.DataFrame(rows)

In [ ]:
def import_models():
    global implicit, tf, Timer, LightGCN, ImplicitCF, prepare_hparams, get_top_k_scored_items
    import implicit
    import tensorflow as tf
    from recommenders.models.deeprec.DataModel.ImplicitCF import ImplicitCF
    from recommenders.models.deeprec.deeprec_utils import prepare_hparams
    from recommenders.models.deeprec.models.graphrec.lightgcn import LightGCN
    from recommenders.utils.python_utils import get_top_k_scored_items
    from recommenders.utils.timer import Timer

    LightGCN.__init__ = new_init
    LightGCN._init_weights = new_init_weights
    LightGCN.recommend_k_items = fast_recommend_k_items


def run(config, seed, run_name):
    out_dir = OUT_ROOT / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    import_models()
    set_config(config, seed)
    print(f"Run {run_name}: {config}", flush=True)
    datasets = load_datasets(config)
    for dataset in datasets.values():
        dataset["implicit"]["RS_strategy"] = {}
    if cf:
        build_cf_data(datasets)
    if lightgcn:
        train_lightgcn(datasets, run_name)
    if content_based:
        build_knn(datasets)
    if cf:
        train_cf(datasets)
    evaluate_models(datasets)
    evaluation_table(datasets).to_csv(out_dir / "evaluation_results.csv", index=False)
    with open(out_dir / "per_user_metrics.pkl", "wb") as f:
        pickle.dump({"config": config, "seed": seed, "metrics": per_user_metrics(datasets)}, f)
    print(f"\nSaved results to {out_dir}", flush=True)

## All runs

The jobs of the grid search, if requested, come first (one job per model family and dataset, seed 0). The final jobs are built in seed-major order (every configuration gets its first seeds before any gets its last ones). LightGCN runs go to a queue per GPU, the other models to a CPU queue. Each run is executed in a child process, with its output written to its log file:

* LightGCN runs only see their GPU, and TensorFlow allocates the GPU memory on demand;
* CPU runs see no GPU and use a single BLAS thread: `implicit` recommends one user at a time, and a multithreaded BLAS makes each call slower while several CPU runs share the machine.

The bar below counts the finished runs. The progress of a run is in its log file.

In [ ]:
def build_grid_jobs(datasets):
    cpu_jobs, gpu_jobs = [], []
    for flag in datasets:
        short = SHORT_NAMES[flag]
        cf_config = {flag: True, "als": True, "bpr": True, "lmf": True,
                     "als_params": GRID["als"], "bpr_params": GRID["bpr"], "lmf_params": GRID["lmf"]}
        cpu_jobs.append((f"grid_search/{short}_cf", cf_config, 0))
        for variant in LIGHTGCN_VARIANTS:
            config = {flag: True, variant: True, "lightgcn_params": GRID["lightgcn"]}
            gpu_jobs.append((f"grid_search/{short}_lightgcn_{variant}", config, 0))
    return cpu_jobs, gpu_jobs


def build_jobs(seeds, datasets):
    cpu_jobs, gpu_jobs = [], []
    for flag in datasets:
        short, best = SHORT_NAMES[flag], BEST[flag]
        cf = {flag: True, "als": True, "bpr": True, "lmf": True,
              "als_params": best["als"], "bpr_params": best["bpr"], "lmf_params": best["lmf"]}
        cpu_jobs += [(f"{short}_cf", cf, seed) for seed in seeds]
        # KNN is deterministic (no training, no sampling): a single run
        cpu_jobs.append((f"{short}_knn", {flag: True, "content_based": True}, 0))
        for variant in LIGHTGCN_VARIANTS:
            config = {flag: True, variant: True, "lightgcn_params": best[variant]}
            gpu_jobs += [(f"{short}_lightgcn_{variant}", config, seed) for seed in seeds]
    # Stable sort: the dataset order is kept within each seed
    cpu_jobs.sort(key=lambda job: job[2])
    gpu_jobs.sort(key=lambda job: job[2])
    return cpu_jobs, gpu_jobs


def child(config, seed, gpu, run_name, log):
    # Runs in a forked process: redirect all output (Python, TensorFlow, C libraries) to the log
    fd = os.open(log, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o644)
    os.dup2(fd, 1)
    os.dup2(fd, 2)
    sys.stdout = sys.stderr = os.fdopen(1, "w", buffering=1, closefd=False)
    # Detach from the notebook kernel: the progress bars of the libraries are written as text to the log
    from IPython.core.interactiveshell import InteractiveShell

    InteractiveShell.clear_instance()
    os.environ["CUDA_VISIBLE_DEVICES"] = gpu
    os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
    try:
        if gpu:
            run(config, seed, run_name)
        else:
            from threadpoolctl import threadpool_limits

            with threadpool_limits(limits=1, user_api="blas"):
                run(config, seed, run_name)
    except BaseException:
        traceback.print_exc()
        sys.stdout.flush()
        os._exit(1)
    sys.stdout.flush()
    os._exit(0)


failed_runs = []


def run_job(run_name, config, seed, gpu):
    run_name = f"{run_name}_seed{seed}"
    if (OUT_ROOT / run_name / "per_user_metrics.pkl").exists():
        print(f"[skip] {run_name}", flush=True)
        return
    log = LOG_DIR / f"{run_name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    print(f"[start] {time.strftime('%H:%M:%S')} {run_name} ({'gpu ' + gpu if gpu else 'cpu'})", flush=True)
    start = time.time()
    process = multiprocessing.get_context("fork").Process(target=child, args=(config, seed, gpu, run_name, str(log)))
    process.start()
    process.join()
    status = "done" if process.exitcode == 0 else "FAILED"
    if process.exitcode != 0:
        failed_runs.append(run_name)
    print(f"[{status}] {time.strftime('%H:%M:%S')} {run_name} ({(time.time() - start) / 60:.1f} min, log {log})", flush=True)


def worker(queue, lock, gpu, progress):
    while True:
        with lock:
            if not queue:
                return
            job = queue.pop(0)
        run_job(*job, gpu)
        progress.update(1)


def run_all(seeds, datasets, gpus, cpu_jobs, grid_search=False):
    cpu_queue, gpu_queue = build_jobs(seeds, datasets)
    if grid_search:
        cpu_grid, gpu_grid = build_grid_jobs(datasets)
        cpu_queue, gpu_queue = cpu_grid + cpu_queue, gpu_grid + gpu_queue
    print(f"{len(cpu_queue)} CPU jobs, {len(gpu_queue)} GPU jobs", flush=True)
    progress = tqdm_auto(total=len(cpu_queue) + len(gpu_queue), desc="Runs")
    cpu_lock, gpu_lock = threading.Lock(), threading.Lock()
    threads = [threading.Thread(target=worker, args=(cpu_queue, cpu_lock, "", progress)) for _ in range(cpu_jobs)]
    threads += [threading.Thread(target=worker, args=(gpu_queue, gpu_lock, gpu, progress)) for gpu in gpus]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    progress.close()
    print("All jobs finished.", flush=True)
    if failed_runs:
        print(f"{len(failed_runs)} FAILED runs (see their logs in {LOG_DIR}): {sorted(failed_runs)}", flush=True)

In [ ]:
run_all(SEEDS, DATASETS, GPUS, CPU_JOBS, RUN_GRID_SEARCH)

## Results

Aggregation of the saved runs:

* **Mean and standard deviation across seeds** of each metric @5, for every dataset and model.
* **Paired tests** on NDCG@5 between pairs of models evaluated on the same test users. For each user, the NDCG@5 of a model is averaged over its seeds; the two models are then compared with a two-sided Wilcoxon signed-rank test over users, and a bootstrap 95% confidence interval of the mean difference is computed by resampling users. The p-values are Holm-corrected over all the tests of the table.

Outputs are written to `results/recommendation_runs/`: `summary_mean_std.csv` and `paired_tests_ndcg.csv`. If the environment variable `DMKD_LATEX_DIR` is set, the two LaTeX tables of the paper are also written to this directory.

In [ ]:
METRICS = {"mrecall": "Recall@5", "mprecision": "Precision@5", "mf1": "F1@5",
           "mndcg": "NDCG@5", "mrr": "MRR@5", "map": "MAP@5"}
PER_USER = {"mndcg": "ndcgs", "mrr": "rrs", "map": "aps",
            "mrecall": "recalls", "mprecision": "precisions", "mf1": "f1s"}

DATASET_NAMES = {"ml-100k": "ml-100k", "LingoRank": "Zeeguu", "Goodreads": "Goodreads", "Tomplay": "Tomplay"}

# Paired comparisons of the paper: (model A, model B, short name)
COMPARISONS = [
    ("LightGCN ada", "LightGCN Xavier ada", "AdaXavier"),
    ("LightGCN bert", "LightGCN Xavier bert", "BertXavier"),
    ("LightGCN ada", "ALS", "AdaALS"),
    ("LightGCN bert", "ALS", "BertALS"),
    ("LightGCN ada", "LightGCN bert", "AdaBert"),
]
N_BOOTSTRAP = 10_000

# Final runs (the grid search is in a subfolder)
RUN_NAME = re.compile(r"^(ml-100k|zeeguu|goodreads|tomplay)_(cf|knn|lightgcn_(?:ada|bert|xavier_ada|xavier_bert))_seed(\d+)$")

### Loading the runs

`runs[(dataset, model)]` maps each seed to the metrics of the model (aggregated and per user).

In [ ]:
def load_runs(seeds):
    runs = defaultdict(dict)
    for pkl in sorted(OUT_ROOT.glob("*/per_user_metrics.pkl")):
        match = RUN_NAME.match(pkl.parent.name)
        if not match:
            continue
        seed = int(match.group(3))
        if seed not in seeds:
            continue
        with open(pkl, "rb") as f:
            content = pickle.load(f)
        for (dataset, model, variant), metrics in content["metrics"].items():
            if model == "_test_users":
                continue
            if model.startswith("KNN") and variant != f"k={KNN_BEST_N[dataset][model.split()[-1]]}":
                continue
            runs[(DATASET_NAMES.get(dataset, dataset), model)][seed] = metrics
    return runs


runs = load_runs(set(SEEDS))
assert runs, f"No finished run in {OUT_ROOT} for seeds {SEEDS}: see the output of run_all above and the logs in {LOG_DIR}"
print(f"{len(runs)} (dataset, model) pairs")

### Mean and standard deviation across seeds

In [ ]:
def summary_table(runs):
    rows = []
    for (dataset, model), seeds in sorted(runs.items()):
        row = {"dataset": dataset, "model": model, "n_seeds": len(seeds),
               "seeds": " ".join(map(str, sorted(seeds)))}
        for key, name in METRICS.items():
            values = np.array([m[key] for m in seeds.values()])
            row[f"{name} mean"] = values.mean()
            row[f"{name} std"] = values.std(ddof=1) if len(values) > 1 else 0.0
            if 0 in seeds:
                row[f"{name} seed0"] = seeds[0][key]
        rows.append(row)
    return pd.DataFrame(rows)


summary = summary_table(runs)
summary.to_csv(OUT_ROOT / "summary_mean_std.csv", index=False)

# Display: mean ± std
display_table = summary[["dataset", "model", "n_seeds"]].copy()
for name in METRICS.values():
    display_table[name] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(summary[f"{name} mean"], summary[f"{name} std"])]
display_table

### Paired tests on NDCG@5

In [ ]:
def per_user_mean(seeds, key="mndcg"):
    """Per-user score averaged over seeds (the users are in the same order in every run)."""
    return np.mean([np.asarray(m[PER_USER[key]]) for m in seeds.values()], axis=0)


def holm(pvalues):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvalues, dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adjusted[i] = min(running, 1.0)
    return adjusted


def paired_tests(runs, rng):
    rows = []
    for dataset in sorted({d for d, _ in runs}):
        for a, b, name in COMPARISONS:
            if (dataset, a) not in runs or (dataset, b) not in runs:
                continue
            x, y = per_user_mean(runs[(dataset, a)]), per_user_mean(runs[(dataset, b)])
            diff = x - y
            nonzero = np.count_nonzero(diff)
            p = wilcoxon(x, y, zero_method="wilcox", alternative="two-sided").pvalue if nonzero else 1.0
            idx = rng.integers(0, len(diff), size=(N_BOOTSTRAP, len(diff)))
            boot = diff[idx].mean(axis=1)
            rows.append({
                "dataset": dataset, "model A": a, "model B": b, "comparison": name,
                "seeds A": len(runs[(dataset, a)]), "seeds B": len(runs[(dataset, b)]),
                "n_users": len(diff), "n_nonzero_diff": nonzero,
                "NDCG@5 A": x.mean(), "NDCG@5 B": y.mean(),
                "mean diff": diff.mean(), "rel. diff (%)": 100 * diff.mean() / y.mean(),
                "CI95 low": np.percentile(boot, 2.5), "CI95 high": np.percentile(boot, 97.5),
                "Wilcoxon p": p,
            })
    tests = pd.DataFrame(rows)
    if len(tests):
        tests["Holm p"] = holm(tests["Wilcoxon p"])
    return tests


tests = paired_tests(runs, np.random.default_rng(0))
tests.to_csv(OUT_ROOT / "paired_tests_ndcg.csv", index=False)
tests.drop(columns=["comparison"])

Relative difference of NDCG@5 (in %, model A vs model B) with the significance level after Holm correction (\* p < 0.05, \*\* p < 0.01, \*\*\* p < 0.001):

In [ ]:
def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""


tests.assign(**{"rel. diff": [f"{d:+.1f}% {stars(p)}" for d, p in zip(tests["rel. diff (%)"], tests["Holm p"])]}) \
    .pivot(index="dataset", columns="comparison", values="rel. diff")[[c for _, _, c in COMPARISONS if c in set(tests.comparison)]]

### LaTeX tables of the paper

Written only if the environment variable `DMKD_LATEX_DIR` is set.

In [ ]:
LATEX_DATASETS = ["Zeeguu", "ml-100k", "Goodreads", "Tomplay"]
LATEX_MODELS = [("ALS", "ALS"), ("BPR", "BPR"), ("LMF", "LMF"),
                ("KNN ada", "KNN ADA$^\\dagger$"), ("KNN bert", "KNN BERT$^\\dagger$"),
                ("LightGCN ada", "LightGCN ADA"), ("LightGCN Xavier ada", "LightGCN Xavier ADA"),
                ("LightGCN bert", "LightGCN BERT"), ("LightGCN Xavier bert", "LightGCN Xavier BERT")]

RESULTS_HEADER = r"""\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}} lccccccc}
\toprule
\rotatebox{45}{\textbf{Dataset}} & \rotatebox{45}{\textbf{Model}} & \rotatebox{45}{\textbf{Recall@5}} & \rotatebox{45}{\textbf{Precision@5}} & \rotatebox{45}{\textbf{F1@5}} & \rotatebox{45}{\textbf{NDCG@5}} & \rotatebox{45}{\textbf{MRR@5}} & \rotatebox{45}{\textbf{MAP@5}} \\
\midrule
"""

TESTS_HEADER = r"""\begin{tabular}{lccccc}
\toprule
 & \multicolumn{2}{c}{LLM init.\ vs Xavier} & \multicolumn{2}{c}{LightGCN vs ALS} & ADA vs BERT \\
\cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-6}
Dataset & ADA & BERT & ADA & BERT & \\
\midrule
"""


def fmt(value):
    """0.0513 -> .0513 (the table is set in \\tiny)."""
    return f"{value:.4f}".lstrip("0")


def latex_results_table(summary):
    """Results table: mean, with the std across seeds as subscript, best value of each dataset in bold."""
    lines = []
    for dataset in LATEX_DATASETS:
        sub = summary[summary.dataset == dataset].set_index("model")
        present = [(m, label) for m, label in LATEX_MODELS if m in sub.index]
        if not present:
            continue
        best = {name: sub[f"{name} mean"].max() for name in METRICS.values()}
        for i, (model, label) in enumerate(present):
            row = sub.loc[model]
            cells = []
            for name in METRICS.values():
                mean, std = row[f"{name} mean"], row[f"{name} std"]
                cell = fmt(mean)
                if np.isclose(mean, best[name]):
                    cell = f"\\mathbf{{{cell}}}"
                if row["n_seeds"] > 1:
                    cell += f"_{{\\pm {fmt(std)}}}"
                cells.append(f"${cell}$")
            first = f"\\multirow{{{len(present)}}}{{*}}{{\\rotatebox{{45}}{{{dataset}}}}}" if i == 0 else ""
            lines.append(f"{first} & {label} & " + " & ".join(cells) + " \\\\")
        lines.append("\\midrule")
    lines[-1] = "\\bottomrule"
    return RESULTS_HEADER + "\n".join(lines) + "\n\\end{tabular*}\n"


def latex_tests_table(tests):
    """Paired-tests table: relative NDCG@5 difference with Holm-corrected significance stars."""
    lines = []
    for dataset in LATEX_DATASETS:
        sub = tests[tests.dataset == dataset].set_index("comparison")
        if sub.empty:
            continue
        cells = []
        for _, _, name in COMPARISONS:
            if name not in sub.index:
                cells.append("--")
                continue
            r = sub.loc[name]
            cells.append(f"${r['rel. diff (%)']:+.1f}^{{{stars(r['Holm p'])}}}$")
        lines.append(f"{dataset} & " + " & ".join(cells) + " \\\\")
    return TESTS_HEADER + "\n".join(lines) + "\n\\bottomrule\n\\end{tabular}\n"


def seeds_note(summary):
    """Number of seeds behind the tables (the deterministic KNN is left out)."""
    stochastic = summary[~summary.model.str.startswith("KNN")]
    counts = stochastic.n_seeds.unique()
    if len(counts) == 1:
        return f"{counts[0]} random seeds"
    full = stochastic.n_seeds.max()
    partial = stochastic[stochastic.n_seeds < full]
    parts = ", ".join(f"{d} {m}: {n}" for d, m, n in partial[["dataset", "model", "n_seeds"]].itertuples(index=False))
    return f"{full} random seeds (fewer for {parts})"


if os.environ.get("DMKD_LATEX_DIR"):
    latex_dir = Path(os.environ["DMKD_LATEX_DIR"])
    latex_dir.mkdir(parents=True, exist_ok=True)
    header = f"% Generated by recommender_system/c_Recommendations.ipynb ({seeds_note(summary)}). Do not edit.\n"
    (latex_dir / "rs_results_table.tex").write_text(header + latex_results_table(summary))
    (latex_dir / "rs_tests_table.tex").write_text(header + latex_tests_table(tests))
    print(f"LaTeX tables written to {latex_dir}")